# Benchmarks

Tianshou's algorithm implementations lead to state-of-the-art results on standard benchmarks.

An efficient parallel implementation for evaluating algorithms on MuJoCo or Atari is in `benchmark/run_benchmark.py`. It can easily be adapted for custom benchmarks as well. The reported results are thus completely reproducible.

The evaluation code uses Tianshou's integration with the [rliable](https://github.com/google-research/rliable) framework, which supports best practices for trustworthy RL evaluation. Each experiment is conducted under 5 random seeds; we report the interquartile mean (IQM) and 95% confidence intervals over these seeds.

In [ ]:
import json
from pathlib import Path

import altair as alt

RESULTS_ROOT = Path("../_static/js")


def load_results(suite: str, environment: str) -> list[dict]:
    results_path = RESULTS_ROOT / suite / "benchmark" / environment / "results.json"
    return json.loads(results_path.read_text(encoding="utf-8"))


def benchmark_chart(suite: str, environment: str) -> alt.LayerChart:
    step_limit = alt.param(
        name="Range",
        value=1_000_000,
        bind=alt.binding_range(min=10_000, max=10_000_000, step=10_000, name="Env step: "),
    )
    base = (
        alt.Chart(alt.Data(values=load_results(suite, environment)))
        .transform_calculate(
            iqm_confidence_lower="datum.iqm_confidence_interval[0]",
            iqm_confidence_upper="datum.iqm_confidence_interval[1]",
        )
        .transform_filter(alt.datum.env_step <= step_limit)
    )
    confidence_interval = base.mark_area(opacity=0.3).encode(
        x=alt.X("env_step:Q", title="Env step"),
        y=alt.Y("iqm_confidence_lower:Q", title="Return Interquartile Mean (5 seeds)"),
        y2="iqm_confidence_upper:Q",
        color="agent:N",
    )
    iqm = base.mark_line().encode(
        x=alt.X("env_step:Q", title="Env step"),
        y=alt.Y("iqm:Q", title="Return Interquartile Mean (5 seeds)"),
        color="agent:N",
        tooltip=[
            alt.Tooltip("env_step:Q", title="Env step"),
            alt.Tooltip("agent:N"),
            alt.Tooltip("iqm:Q", title="Return"),
        ],
    )
    return (
        (confidence_interval + iqm)
        .add_params(step_limit)
        .properties(
            title=environment,
            width=800,
            height=400,
        )
    )

In [ ]:
benchmark_chart("mujoco", "Ant-v4")